In [ ]:
# 0. Repository and paths -- every location is read from config.yaml (see README), none is typed here.
import os
import sys


def _find_repo():
    for c in (os.environ.get("QCT3D_REPO", ""), os.getcwd(), os.path.dirname(os.getcwd()),
              "/content/qct3d-saturation"):
        if c and os.path.isfile(os.path.join(c, "src", "qct3d", "config.py")):
            return os.path.abspath(c)
    raise RuntimeError("Cannot find the repository. Clone it first (README, 'Running on Colab') "
                       "or set the environment variable QCT3D_REPO to its folder.")


REPO_ROOT = _find_repo()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from qct3d.config import load_config  # noqa: E402

P = load_config()

In [ ]:
# Overnight run: encoder saturation, India label efficiency, and RSNA -> India
# few-shot transfer
#
# Cells 1-7 below are nb2's, verbatim, except for one addition to `Reducer`:
# `set_encoder_norm()`, a non-affine BatchNorm between the learned projection and
# the tanh. It adds no parameters and is a no-op on the PCA reducers.
#
# Everything is resumable. If the runtime dies or the budget runs out, re-run the
# notebook and it continues from the log on Drive.
# 1. Drive
import os

try:
    from google.colab import drive
    drive.mount(P["drive_mount"])
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("not in Colab -- point OUTPUT_DIR at wherever Phase A wrote its caches.")
if IN_COLAB and not os.path.isdir(P["drive_root"]):
    raise RuntimeError("Drive did not finish mounting -- re-run this cell.")
print("ok")

In [ ]:
# 2. Installs
#
# Phase B needs far less than Phase A: no MONAI, no pydicom, no SimpleITK. Only PennyLane
# (the circuits), torch (the graph they live in), and pyarrow (to read the feature tables).
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pennylane>=0.35,<0.50", "pyarrow>=14", "pandas>=2.1", "scikit-learn>=1.3"],
               check=False)

import pennylane as qml
import torch

print(f"pennylane {qml.__version__}   torch {torch.__version__}")
print("NOTE: this notebook does not need a GPU. A CPU runtime is the right choice --")
print("      it leaves your GPU quota for Phase A, where it actually matters.")

In [ ]:
# 3. CONFIGURATION
import os

# Must match Phase A.
OUTPUT_DIR   = P["rsna_output_dir"]
FEATURE_DIR  = os.path.join(OUTPUT_DIR, "features")
SPATIAL_DIR  = os.path.join(OUTPUT_DIR, "spatial_maps")

# ---- the grid -------------------------------------------------------------------------
# Comment out any backbone you have not extracted yet; the notebook reports what it found
# and runs the rest rather than failing.
BACKBONES_TO_RUN = [
    "B1_swinunetr_suprem",
    "B2_segresnet_suprem",
    "B3_resnet18_medicalnet",
    # "B4_swinunetr_ssl5050",
]
# Q3_quanv is REMOVED here on purpose, and it is not an oversight.
#
# Every arm this notebook fits is a POOLED one -- E0 audits the pooled families, and the E6
# and E7 curves use CURVE_FAMILY, which is pooled too. Leaving Q3_quanv in this list sets
# NEED_SPATIAL in the next cell, which then pulls roughly 11,600 individual .npy files off
# Drive (about 500 MB, and Drive is latency-bound on small files) to build maps that nothing
# below ever reads. On a Colab runtime that is easily half an hour of the night spent on
# nothing. Put it back if you add a spatial arm to the curves.
# E0 now measures the POOLED and the SPATIAL arm in the same harness, on the same splits and
# the same seeds, because that comparison is the paper's central claim and the two numbers
# previously came from different holdouts. That needs the cached spatial maps, so Q3_quanv is
# back in this list. It costs a one-off Drive read of ~11,600 small files (~500 MB) at the
# start of the run. Set E0_INCLUDE_SPATIAL = False below to skip that.
E0_INCLUDE_SPATIAL = False  # the sweep is pooled-only; no spatial maps needed
FAMILIES_TO_RUN = (["Q1_sel", "Q2_reupload", "Q4_ttn"]
                   + (["Q3_quanv"] if E0_INCLUDE_SPATIAL else []))

# ---- the reducer ----------------------------------------------------------------------
# How a 768-2304-d feature vector becomes N_QUBITS rotation angles matters MORE than the
# choice of ansatz. Running one and presenting it as neutral is the easiest way to mislead
# yourself, so the grid runs on the primary and the finalists are re-run on all three.
#
#   "pca_supervised"  PCA on the training fold, then the components most correlated with
#                     the label. Nothing outside the adapter is trainable.
#   "trainable"       a trainable Linear(d -> n_qubits) trained jointly with the circuit.
#                     Highest performing; holds most of the parameters, which is exactly
#                     why the classical control shares it and only the TAIL is matched.
#   "pca"             plain top-variance PCA. Label-blind, and the honest floor.
PRIMARY_REDUCER  = "pca_supervised"
FINALIST_REDUCERS = ["pca_supervised", "trainable", "pca"]

# ---- targets --------------------------------------------------------------------------
PRIMARY_TARGET   = "any"          # ICH presence: the powered endpoint
SECONDARY_TARGETS = ["intraparenchymal", "subdural", "subarachnoid", "intraventricular",
                     "epidural"]
RUN_SECONDARY_ON_FINALIST_ONLY = True    # subtypes are exploratory; see the outline
MIN_POSITIVES    = 30

# ---- circuit --------------------------------------------------------------------------
N_QUBITS   = 4       # must be a power of two: Q4's tree ansatz needs one
DEPTH      = 3       # SEL layers (Q1/Q3) and re-uploads (Q2); Q4's depth is fixed by the tree
DIFF_METHOD = "backprop"      # measured against adjoint in the benchmark cell below
INIT_STD   = 0.01             # near-identity init: barren-plateau mitigation

# ---- optimisation ---------------------------------------------------------------------
EPOCHS          = 80
EPOCHS_SPATIAL  = 25      # Q3 sees n_positions times more circuit evaluations per epoch
BATCH           = 128
BATCH_SPATIAL   = 32
LR              = 0.05
ANGLE_L2_BETA   = 1e-4    # penalty on the ROTATION ANGLES specifically
PROJ_WEIGHT_DECAY = 1e-3  # on the reducer/projection only
FOCAL_ALPHA, FOCAL_GAMMA = 0.25, 2.0

# ---- protocol ---------------------------------------------------------------------------
CV_FOLDS      = 5
HOLDOUT_FRAC  = 0.2
TARGET_SENSITIVITY = 0.85
N_BOOTSTRAP   = 2000
SEED          = 12345
NISQ_P        = 0.01
FORCE_RERUN   = False

RESULTS_JSONL = os.path.join(OUTPUT_DIR, "phaseb_results.jsonl")
RESULTS_CSV   = os.path.join(OUTPUT_DIR, "phaseb_leaderboard.csv")
FIG_DIR       = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

if N_QUBITS & (N_QUBITS - 1):
    raise SystemExit(f"N_QUBITS must be a power of two (Q4's tree ansatz halves the wires "
                     f"at every level); got {N_QUBITS}")

CFG = dict(n_qubits=N_QUBITS, depth=DEPTH, diff_method=DIFF_METHOD, init_std=INIT_STD,
           epochs=EPOCHS, epochs_spatial=EPOCHS_SPATIAL, batch=BATCH,
           batch_spatial=BATCH_SPATIAL, lr=LR, angle_l2_beta=ANGLE_L2_BETA,
           projection_weight_decay=PROJ_WEIGHT_DECAY, focal_alpha=FOCAL_ALPHA,
           focal_gamma=FOCAL_GAMMA, cv_folds=CV_FOLDS, holdout_frac=HOLDOUT_FRAC,
           seed=SEED, n_bootstrap=N_BOOTSTRAP, target_sensitivity=TARGET_SENSITIVITY,
           predict_batch=256, dataset="rsna-ich")
CFG["signature"] = "|".join([
    f"q={N_QUBITS}", f"d={DEPTH}", f"diff={DIFF_METHOD}", f"ep={EPOCHS}/{EPOCHS_SPATIAL}",
    f"lr={LR}", f"bs={BATCH}/{BATCH_SPATIAL}", f"beta={ANGLE_L2_BETA}",
    f"wd={PROJ_WEIGHT_DECAY}", f"focal=({FOCAL_ALPHA},{FOCAL_GAMMA})",
    f"folds={CV_FOLDS}", f"holdout={HOLDOUT_FRAC}", f"seed={SEED}",
])
print("config signature:", CFG["signature"])
print(f"grid: {len(BACKBONES_TO_RUN)} backbones x {len(FAMILIES_TO_RUN)} families "
      f"= {len(BACKBONES_TO_RUN)*len(FAMILIES_TO_RUN)} cells")

In [ ]:
# 4. Load every backbone's cached features, and the spatial maps the Q3 arm needs
#
# The three backbones do NOT share a feature width (768/128/512 per window x 3 windows),
# which is fine -- each is its own row of the grid and the reducer absorbs the difference.
# What they MUST share is the set of series, because otherwise a "difference between
# backbones" is partly a difference between cohorts. The intersection is taken explicitly
# and reported, rather than each backbone being scored on whatever it happens to have.
import concurrent.futures as _cf
import time as _time

import numpy as np
import pandas as pd

SPATIAL_LOAD_WORKERS = 16     # Drive is latency-bound; this is round trips, not bytes

LABEL_COLS = ["any", "epidural", "intraparenchymal", "intraventricular", "subarachnoid",
              "subdural"]

TABLES, MISSING = {}, []
for b in BACKBONES_TO_RUN:
    p = os.path.join(OUTPUT_DIR, f"features_{b}.parquet")
    if not os.path.exists(p):
        MISSING.append(b)
        continue
    TABLES[b] = pd.read_parquet(p)
    print(f"{b:<26} {len(TABLES[b]):>6,} series x "
          f"{sum(c.startswith('f_') for c in TABLES[b].columns):>5} features")
if MISSING:
    print(f"\nNOT FOUND (skipped): {MISSING}")
    print("  run nb1_phaseA_extract.ipynb with BACKBONE_ID set to each of these first.")
if not TABLES:
    raise SystemExit("no feature tables found -- Phase A has not produced anything yet.")

# The intersection is what protects this notebook from a real hazard in Phase A: cell 15
# there admits any series whose progress record says "ok" under the current preprocessing
# signature, WITHOUT checking that the series is in the current manifest. B1 was extracted
# once before on an earlier 86-series cohort under an identical signature, so 68 of those
# came along into features_B1_swinunetr_suprem.parquet. Because Phase A merges metadata with
# a LEFT join, those rows carry NaN for every label and for `patient` -- and a NaN in
# `patient` would silently corrupt the patient-grouped split that the whole protocol rests
# on. Taking the intersection FIRST removes them; the assertions below prove it.
common = None
for b, t in TABLES.items():
    s = set(t["series"])
    common = s if common is None else (common & s)
common = sorted(common)
print(f"\nseries present in ALL {len(TABLES)} backbone(s): {len(common):,}")
for b, t in TABLES.items():
    if len(t) != len(common):
        _extra = sorted(set(t["series"]) - set(common))
        _nan = int(t[t["series"].isin(_extra)]["any"].isna().sum()) if "any" in t.columns else 0
        print(f"  {b}: dropping {len(t)-len(common):,} series not shared by every backbone"
              + (f"   ({_nan:,} of them have no label at all -- out-of-cohort leftovers)"
                 if _nan else ""))
if len(common) < 50:
    raise SystemExit(f"only {len(common)} shared series -- extract more before running the grid.")

X_BY_BACKBONE, M_BY_BACKBONE = {}, {}
meta = None
for b, t in TABLES.items():
    t = t[t["series"].isin(common)].sort_values("series").reset_index(drop=True)
    fcols = [c for c in t.columns if c.startswith("f_")]
    X_BY_BACKBONE[b] = t[fcols].to_numpy(dtype=np.float32)
    if meta is None:
        meta = t[["series", "patient", "study"] +
                 [c for c in LABEL_COLS if c in t.columns]].copy()

# Every downstream split is grouped by these. A NaN here does not raise -- it becomes the
# string "nan", silently collapsing unrelated patients into ONE group and leaking scans of
# different people across the train/test boundary. Checked rather than trusted.
if meta["patient"].isna().any():
    raise SystemExit(
        f"{int(meta['patient'].isna().sum()):,} of {len(meta):,} rows have no patient id. "
        f"They are series that are in a feature table but not in manifest.parquet, so the "
        f"metadata merge in Phase A left them empty. Every patient-grouped split would be "
        f"wrong. Fix the feature tables (restrict them to manifest.parquet) before running.")
for _lc in [PRIMARY_TARGET] + SECONDARY_TARGETS:
    if _lc in meta.columns and meta[_lc].isna().any():
        raise SystemExit(f"{int(meta[_lc].isna().sum()):,} rows have no {_lc!r} label; "
                         f"same cause and same fix as above.")

SERIES_ALL = meta["series"].to_numpy()
GROUPS_ALL = meta["patient"].astype(str).to_numpy()   # group by PATIENT, never by series
assert len(set(GROUPS_ALL)) > 1, "every series resolved to one patient group"
print(f"patients: {len(set(GROUPS_ALL)):,}  ({len(SERIES_ALL)/max(len(set(GROUPS_ALL)),1):.2f} "
      f"series per patient)")

# ---- spatial maps (Q3 only) -------------------------------------------------------------
NEED_SPATIAL = any(f == "Q3_quanv" for f in FAMILIES_TO_RUN)
if NEED_SPATIAL:
    for b in list(X_BY_BACKBONE):
        d = os.path.join(SPATIAL_DIR, b)
        if not os.path.isdir(d):
            print(f"\nNO SPATIAL MAPS for {b} at {d}")
            print("  Q3 is the deep-placement arm and cannot run from pooled vectors. Either")
            print("  re-run Phase A for this backbone with SAVE_SPATIAL_MAP=True, or drop")
            print("  'Q3_quanv' from FAMILIES_TO_RUN.")
            continue
        # SERIAL LOADING HERE IS THE SAME MISTAKE PHASE A MADE. This is 3,889 individual
        # .npy files per backbone, ~11,700 across three, and on a Drive FUSE mount it is the
        # per-file round trip that costs -- not the bytes. One at a time that is 20 minutes
        # to 3 hours before a single circuit trains; a thread pool makes it a couple of
        # minutes. Order is preserved by index, NOT by completion order: these rows are
        # aligned with meta["series"] and with X_BY_BACKBONE, so shuffling them would pair
        # every spatial map with the wrong label.
        _ids = list(meta["series"])
        _paths = [os.path.join(d, f"{s}.npy") for s in _ids]
        arrs = [None] * len(_paths)

        def _one(i, _p=_paths):
            try:
                return i, np.load(_p[i])
            except Exception:
                return i, None

        _t0 = _time.time()
        with _cf.ThreadPoolExecutor(max_workers=SPATIAL_LOAD_WORKERS) as _ex:
            for _n, (_i, _a) in enumerate(_ex.map(_one, range(len(_paths))), 1):
                arrs[_i] = _a
                if _n % 500 == 0 or _n == len(_paths):
                    print(f"\r  {b}: loaded {_n:,}/{len(_paths):,} maps "
                          f"({_time.time()-_t0:.0f}s)   ", end="", flush=True)
        print()
        _n_missing = sum(1 for _a in arrs if _a is None)
        if _n_missing:
            print(f"  {b}: {_n_missing:,} spatial map(s) missing on disk")
        ok = _n_missing == 0
        if ok and arrs:
            _shapes = {a.shape for a in arrs}
            if len(_shapes) != 1:
                print(f"  {b}: spatial maps have {len(_shapes)} different shapes {list(_shapes)[:3]}"
                      f" -- Q3 skipped for this backbone")
                ok = False
        if ok and arrs:
            M_BY_BACKBONE[b] = np.stack(arrs).astype(np.float32)
            print(f"spatial maps {b:<26} {M_BY_BACKBONE[b].shape}  "
                  f"({M_BY_BACKBONE[b].nbytes/1e6:.0f} MB, "
                  f"{int(np.prod(M_BY_BACKBONE[b].shape[2:]))} positions/series)")
        else:
            print(f"spatial maps {b}: incomplete -- Q3 will be skipped for this backbone")

# ---- targets ----------------------------------------------------------------------------
print(f"\n{'target':<22}{'positives':>11}{'prevalence':>13}{'majority acc':>15}")
TARGETS = {}
for t in [PRIMARY_TARGET] + SECONDARY_TARGETS:
    if t not in meta.columns:
        continue
    y = (meta[t].to_numpy() > 0).astype(int)
    npos = int(y.sum())
    flag = ""
    if npos < MIN_POSITIVES or (len(y) - npos) < MIN_POSITIVES:
        flag = f"  <-- below MIN_POSITIVES={MIN_POSITIVES}, skipped"
    else:
        TARGETS[t] = y
    print(f"  {t:<20}{npos:>11,}{y.mean():>13.4f}{max(y.mean(),1-y.mean()):>15.3f}{flag}")
if PRIMARY_TARGET not in TARGETS:
    raise SystemExit(f"the primary target {PRIMARY_TARGET!r} does not have enough positives.")
print("\n  'majority acc' is what you score by always answering the majority class. Any")
print("  accuracy at or below it is worth nothing, which is why every accuracy below is")
print("  printed next to it.")

In [ ]:
# 5. The four quantum-circuit families, the adapters, and the training loop
#
# Moved verbatim to src/qct3d/models.py and imported here into the notebook namespace.
# Every later cell sees exactly the names this cell used to define.
from qct3d.models import *  # noqa: F401,F403

In [ ]:
# 6. The protocol: patient-grouped splits, the three arms, and the paired comparison
#
# Moved verbatim to src/qct3d/protocol.py and imported here into the notebook namespace.
# Every later cell sees exactly the names this cell used to define.
from qct3d.protocol import *  # noqa: F401,F403

In [ ]:
# 7. Parameter budgets, and a self-test that proves gradients reach the rotation angles
#
# ASSERTED, because each failure means the cell is broken: the angles must MOVE during
# training (if they do not, no gradient is reaching the circuit and every quantum number
# below is an untrained random circuit's output), the loss must go DOWN, and the outputs
# must be finite.
#
# REPORTED, not asserted: the AUROC on a planted signal. How high it gets depends on the
# epoch count, the learning rate and the qubit count, so asserting a floor on it fires
# when nothing is wrong -- an alarm that goes off spuriously is worse than no alarm.
import numpy as np
import torch

print(f"{'family':<14}{'role':<46}{'tail':>6}{'hidden':>8}{'control':>9}{'err':>7}")
print("-" * 92)
BUDGETS = {}
for fam in FAMILIES_TO_RUN:
    probe = AdapterModel("quantum", Reducer("pca", 64, N_QUBITS), N_QUBITS, DEPTH,
                         family=fam, diff_method="backprop")
    tail = probe.tail_params()
    hid, err, cpar = match_hidden_width(N_QUBITS, tail)
    BUDGETS[fam] = {"tail": tail, "hidden": hid, "control_params": cpar, "err": err}
    role = QUANTUM_FAMILIES[fam]["role"]
    print(f"{fam:<14}{role[:44]:<46}{tail:>6}{hid:>8}{cpar:>9}{err:>6.1%}")
    if err > 0.10:
        print(f"  *** {fam}: outside the +/-10% matching band. With this few parameters the "
              f"hidden width is quantised too coarsely -- report the exact counts in the paper "
              f"rather than claiming a match. ***")
print(f"\nlinear probe floor: the full feature vector + 1 bias, no adapter, no reduction")


def _self_test(fam):
    rng = np.random.default_rng(0)
    n, d, k = 600, 128, 20
    L = rng.standard_normal((d, k)).astype("float32")
    Z = rng.standard_normal((n, k)).astype("float32")
    X = np.maximum(Z @ L.T + 0.5 * rng.standard_normal((n, d)), 0).astype("float32")
    s = Z @ rng.standard_normal(k) + rng.standard_normal(n)
    y = (s >= np.quantile(s, 0.86)).astype(int)
    tr, te = np.arange(450), np.arange(450, n)
    red = Reducer("trainable", d, N_QUBITS, seed=0).fit(X[tr], y[tr], seed=0)
    m = AdapterModel("quantum", red, N_QUBITS, DEPTH, family=fam,
                     diff_method=DIFF_METHOD, init_std=INIT_STD, seed=0)
    Ztr, Zte = red.transform_np(X[tr]), red.transform_np(X[te])
    before = m.q.weights.detach().clone()
    Xt = torch.as_tensor(Ztr, dtype=torch.float32)
    yt = torch.as_tensor(y[tr], dtype=torch.float32)
    with torch.no_grad():
        l0 = float(focal_loss(m(Xt), yt, FOCAL_ALPHA, FOCAL_GAMMA))
    train_adapter(m, Ztr, y[tr], epochs=40, lr=LR, batch=BATCH, beta=ANGLE_L2_BETA,
                  weight_decay=PROJ_WEIGHT_DECAY, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, seed=0)
    with torch.no_grad():
        l1 = float(focal_loss(m(Xt), yt, FOCAL_ALPHA, FOCAL_GAMMA))
    moved = float((m.q.weights.detach() - before).abs().max())
    p = predict_proba(m, Zte)
    return moved, l0, l1, fast_auroc(y[te], p), bool(np.isfinite(p).all())


print("\nself-test on synthetic features with a planted signal (~14% prevalence):")
for fam in [f for f in FAMILIES_TO_RUN if not QUANTUM_FAMILIES[f]["spatial"]]:
    mv, l0, l1, auc, fin = _self_test(fam)
    print(f"  {fam:<14} angles moved {mv:.5f} | focal {l0:.5f} -> {l1:.5f} | "
          f"AUROC {auc:.3f} | finite {fin}")
    assert fin, f"{fam}: non-finite probabilities -- the circuit is broken."
    assert mv > 1e-6, (
        f"{fam}: the rotation angles did not move during training (max change {mv:.2e}). No "
        f"gradient is reaching the circuit, so every number this notebook produces for this "
        f"family would be an untrained random circuit's output. Check DIFF_METHOD.")
    assert l1 < l0, (f"{fam}: the focal loss did not decrease ({l0:.5f} -> {l1:.5f}). The "
                     f"optimiser is making no progress; check LR (currently {LR}).")
print("  -> gradients reach the rotation angles and the loss falls: the circuits train.")

In [ ]:
# ==============================================================================================
# OV-1. OVERNIGHT CONFIGURATION -- budgets, resume, and the one knob the whole paper turns on
# ==============================================================================================
# This notebook reuses nb2's cells 1-7 VERBATIM above (config, feature loading, the circuit
# families, the adapters, the training loop, the protocol, the budgets). Nothing about the
# models changes, so every number here is directly comparable with the Phase-B grid.
#
# What is added below, in priority order, is three experiments:
#
#   E0  SATURATION AUDIT + ENCODER-NORM PARITY.
#       Q1/Q2/Q4 with reducer mode "trainable" have a learned nn.Linear feeding tanh and NO
#       normalisation. Q3 has a learned conv feeding tanh WITH a non-affine BatchNorm, added
#       when Q3 was found stuck at AUROC exactly 0.500. The documented cause was weight growth
#       under AdamW driving pre-tanh magnitude to 30-45, where tanh is flat to machine
#       precision. That mechanism is input-independent, so the trainable pooled arms are
#       exposed to it too -- and the 9,261-parameter-matched comparison that the "placement
#       dominates ansatz" claim rests on is exactly Q1_trainable vs Q3. E0 measures pre-tanh
#       magnitude for every arm and re-runs the pooled arms with the same normalisation.
#
#       NOTE, and this matters for how you write it up: the PCA reducers cannot saturate.
#       They have no trainable projection and their components are divided by their own std
#       at fit time, so pre-tanh is O(1) by construction. Table 1's Q1 = 0.749 uses
#       pca_supervised and is NOT at risk. The exposed cell is Q1 with mode "trainable".
#
#   E6  INDIA LABEL-EFFICIENCY CURVE. Train from scratch on India only at k local labels.
#       This is the direct analogue, on real clinical data, of the MNIST experiment behind
#       the published claim that quantum adapters degrade more gracefully when labels are
#       scarce. normal_vs_abnormal carries 859 positives and is the powered target.
#
#   E7  RSNA -> INDIA FEW-SHOT TRANSFER. Train the adapter on RSNA, then adapt it on k India
#       labels. Zero-shot is the 0.490 cell already measured; this asks how many local labels
#       buy it back, and whether quantum buys it back faster than a parameter-matched
#       classical tail.
#
# Everything is RESUMABLE and BUDGETED. Each unit of work writes one JSON line as it
# finishes, keyed by everything that defines it, so re-running skips what is done. When
# WALL_BUDGET_MIN is spent the notebook stops cleanly between units and tells you what is
# left. Run it again tomorrow night to continue.
import json
import os
import time

import numpy as np

# ---- where India lives -------------------------------------------------------------------
INDIA_DIR = P["external_output_dir"]
INDIA_LABEL_SOURCE = "auto"      # "auto" finds matched_table.parquet in the usual folders

# ---- what to run, in order. Drop names to skip a stage. -----------------------------------
STAGES = []   # this notebook runs the sweep below, not the E-stages

# ---- wall clock -------------------------------------------------------------------------
WALL_BUDGET_MIN = 420            # 7 h. The loop stops cleanly between units, never mid-fit.
OV_LOG = os.path.join(OUTPUT_DIR, "overnight_results.jsonl")

# ---- E0 -----------------------------------------------------------------------------------
E0_FAMILIES = ["Q1_sel", "Q2_reupload", "Q4_ttn"]
E0_REDUCERS = ["trainable"]      # the only reducer that CAN saturate
E0_NORMS    = [False, True]      # False reproduces the current grid; True is the parity run
E0_BACKBONES = None              # None = every backbone in BACKBONES_TO_RUN
E0_SEEDS    = [12345, 1, 2, 3, 4]
# FIVE SEEDS, because E0 is now the paper's central table and its central claim is a
# comparison. One seed gives point estimates and no way to say whether pooled-vs-spatial is a
# real difference or training noise -- which is exactly the criticism the whole project is
# open to. Five is cheap here: a pooled fit is 12 s and a spatial fit a few minutes.
E0_SPATIAL_FAMILY = "Q3_quanv" if globals().get("E0_INCLUDE_SPATIAL", False) else None
# The spatial arm goes through the SAME splits, seeds and scoring as the pooled arms. Its
# 0.867 in the Phase-B grid came from a different holdout, so the two were never directly
# comparable -- and "placement dominates ansatz" is a comparison between exactly those two
# numbers. This makes it one measurement.

# ---- E6 / E7 ------------------------------------------------------------------------------
# Budgets are in LABELLED SERIES, drawn patient-grouped. 0 means zero-shot (E7 only).
LABEL_BUDGETS   = [25, 50, 100, 200, 400, 900]
LABEL_BUDGETS_T = [0, 10, 25, 50, 100, 200, 400, 900]
N_DRAWS         = 50             # independent draws per budget; the curve is the result
#
# MEASURED COST, so you can size this yourself. On 2 CPU threads, one fit takes:
#   quantum   4.1 s at n=25,  12.5 s at n=900,  49 s at n=3,889
#   classical 0.1 s          1.0 s              4.0 s
#   linear    0.01 s         0.14 s             0.8 s
# The quantum arm dominates and the whole default run below is on the order of 3-5 hours
# for all three backbones at 50 draws. If you want it shorter, cut N_DRAWS or CURVE_BACKBONES
# first -- never the budgets, because the SHAPE of the curve is the result.
CURVE_BACKBONES = ["B1_swinunetr_suprem", "B2_segresnet_suprem", "B3_resnet18_medicalnet"]
# Three backbones is three independent replications of the same curve. That is worth far
# more to a reviewer than one curve with tighter error bars, and it is affordable here.
CURVE_FAMILY    = "Q1_sel"       # pooled arm: comparable to the published MNIST experiment
CURVE_REDUCER   = "trainable"
CURVE_ARMS      = ["quantum", "classical", "linear_probe"]
CURVE_TARGETS_INDIA = ["abnormal_any", "hemorrhage"]   # powered first, then the clinical one
ADAPT_MODES     = ["tail", "tail_plus_projection"]     # E7 only; "readout" is added free

ENCODER_NORM_DEFAULT = True      # what E6/E7 use, so their arms are never the saturated ones

os.makedirs(os.path.dirname(OV_LOG) or ".", exist_ok=True)
_T0 = time.time()


def budget_left_min():
    return WALL_BUDGET_MIN - (time.time() - _T0) / 60.0


def out_of_budget(need_min=2.0):
    return budget_left_min() < need_min


def ov_done_keys(path=None):
    """Every unit key already on the log. The log is append-only; last line wins."""
    path = path or OV_LOG
    done = {}
    if os.path.exists(path):
        with open(path) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if r.get("key"):
                    done[r["key"]] = r
    return done


def ov_record(key, **fields):
    rec = {"key": key, "t": time.strftime("%Y-%m-%d %H:%M:%S"), **fields}
    with open(OV_LOG, "a") as fh:
        fh.write(json.dumps(rec, default=float) + "\n")
    return rec


OV_DONE = ov_done_keys()
print("=" * 94)
print("OVERNIGHT RUN")
print("=" * 94)
print(f"  stages          : {STAGES}")
print(f"  wall budget     : {WALL_BUDGET_MIN} min")
print(f"  results log     : {OV_LOG}")
print(f"  already done    : {len(OV_DONE):,} unit(s) -- these will be skipped")
print(f"  label budgets   : E6 {LABEL_BUDGETS}  |  E7 {LABEL_BUDGETS_T}")
print(f"  draws per point : {N_DRAWS}")
print(f"  curve backbones : {CURVE_BACKBONES}")
print("=" * 94)


In [ ]:
# ==============================================================================================
# OV-3. THE PRIMITIVES the three experiments share
# ==============================================================================================
# Everything here is built from nb2's own Reducer / AdapterModel / train_adapter, so an arm
# fitted here is the same object the Phase-B grid fits. Nothing is reimplemented.
import copy

import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold

OV_CFG = dict(CFG) if isinstance(globals().get("CFG"), dict) else {}
OV_CFG.setdefault("n_qubits", N_QUBITS)
OV_CFG.setdefault("depth", globals().get("DEPTH", 3))
OV_CFG.setdefault("diff_method", DIFF_METHOD)
OV_CFG.setdefault("init_std", INIT_STD)


def fixed_eval_split(y, groups, frac=0.25, seed=12345):
    """ONE evaluation set, shared by every budget, draw and arm.

    If each point on a learning curve got its own test set, the curve would mix label-budget
    effects with test-set sampling, and the two are the same order of magnitude at these
    prevalences. Patient-grouped and label-stratified.
    """
    y = np.asarray(y).astype(int)
    n_splits = max(2, int(round(1.0 / max(frac, 1e-6))))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    pool_idx, eval_idx = next(iter(sgkf.split(np.zeros(len(y)), y, groups)))
    return np.asarray(pool_idx), np.asarray(eval_idx)


def draw_budget(y_pool, g_pool, k, seed):
    """k labelled series drawn WHOLE PATIENTS at a time, keeping both classes present.

    Sampling series independently would split a patient across the labelled and unlabelled
    sets, which is not what "we labelled k scans at the new hospital" means and which leaks
    patient identity into the budget.
    """
    rng = np.random.default_rng(seed)
    y_pool = np.asarray(y_pool).astype(int)
    g_pool = np.asarray(g_pool)
    if k >= len(y_pool):
        return np.arange(len(y_pool))
    pats = rng.permutation(np.unique(g_pool))
    idx_by_pat = {p: np.flatnonzero(g_pool == p) for p in pats}
    take, chosen = [], 0
    for p in pats:
        ii = idx_by_pat[p]
        if chosen + len(ii) > k and chosen > 0:
            continue
        take.append(ii)
        chosen += len(ii)
        if chosen >= k:
            break
    sel = np.concatenate(take) if take else np.arange(min(k, len(y_pool)))
    if len(np.unique(y_pool[sel])) < 2:
        # A single-class budget cannot train anything. Swap in the nearest patient carrying
        # the missing class rather than returning a draw that silently produces NaN.
        missing = 1 - int(y_pool[sel][0])
        cand = [p for p in pats if (y_pool[idx_by_pat[p]] == missing).any()]
        if cand:
            sel = np.concatenate([sel, idx_by_pat[cand[0]]])
    return np.unique(sel)


def make_pooled_arm(arm, d_in, seed, encoder_norm=True, family=None, reducer_mode=None):
    """A pooled (Q1-style) adapter, or None for the linear probe."""
    if arm == "linear_probe":
        return None
    family = family or CURVE_FAMILY
    reducer_mode = reducer_mode or CURVE_REDUCER
    red = Reducer(reducer_mode, d_in, OV_CFG["n_qubits"], seed=seed)
    if hasattr(red, "set_encoder_norm"):
        red.set_encoder_norm(encoder_norm)
    probe = AdapterModel("quantum", red, OV_CFG["n_qubits"], OV_CFG["depth"], family=family,
                         diff_method=OV_CFG["diff_method"], init_std=OV_CFG["init_std"],
                         seed=seed)
    tail = probe.tail_params()
    if arm == "quantum":
        return probe
    hidden, _e, _p = match_hidden_width(OV_CFG["n_qubits"], tail)
    return AdapterModel("classical", red, OV_CFG["n_qubits"], OV_CFG["depth"], family=family,
                        hidden=hidden, diff_method=OV_CFG["diff_method"],
                        init_std=OV_CFG["init_std"], seed=seed)


def make_spatial_arm(arm, c_in, seed, family="Q3_quanv"):
    """Q3's deep-placement arm, parameter-matched the way the Phase-B grid matches it."""
    if arm == "linear_probe":
        return None
    probe = SpatialAdapter("quantum", c_in, OV_CFG["n_qubits"], OV_CFG["depth"],
                           family=family, diff_method=OV_CFG["diff_method"],
                           init_std=OV_CFG["init_std"], seed=seed)
    if arm == "quantum":
        return probe
    hidden, _e, _p = match_hidden_width(OV_CFG["n_qubits"], probe.tail_params())
    return SpatialAdapter("classical", c_in, OV_CFG["n_qubits"], OV_CFG["depth"],
                          hidden=hidden, family=family, diff_method=OV_CFG["diff_method"],
                          init_std=OV_CFG["init_std"], seed=seed)


def fit_arm(arm, Xtr, ytr, seed, epochs=None, encoder_norm=True, model=None,
            adapt_mode=None, lr=None, family=None, reducer_mode=None, Mtr=None):
    """Fit one arm and return (model_or_sklearn, fitted_reducer_or_scaler).

    `model` non-None means ADAPT an existing model rather than build a new one, and
    `adapt_mode` says how much of it is unfrozen.
    """
    ytr = np.asarray(ytr).astype(int)
    if arm == "linear_probe":
        sc = StandardScaler().fit(Xtr)
        lrm = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0)
        lrm.fit(sc.transform(Xtr), ytr)
        return lrm, sc

    if model is None:
        if Mtr is not None:
            # The SPATIAL arm. It carries its own conv+BatchNorm projection, so encoder_norm
            # does not apply and there is no Reducer to fit -- the maps go in raw, exactly as
            # the Phase-B grid feeds them.
            model = make_spatial_arm(arm, Mtr.shape[1], seed, family=family)
            for p in model.parameters():
                p.requires_grad_(True)
            train_adapter(model, Mtr, ytr, epochs=epochs or EPOCHS_SPATIAL, lr=lr or LR,
                          batch=BATCH_SPATIAL, beta=ANGLE_L2_BETA,
                          weight_decay=PROJ_WEIGHT_DECAY, alpha=FOCAL_ALPHA,
                          gamma=FOCAL_GAMMA, seed=seed,
                          reducer_prefixes=("reducer.", "proj."))
            return model, None
        # family/reducer_mode were NOT forwarded here in the first version of this notebook,
        # so every E0 row silently used CURVE_FAMILY regardless of the family it was labelled
        # with. That is why Q1/Q2/Q4 came back identical to six decimal places.
        model = make_pooled_arm(arm, Xtr.shape[1], seed, encoder_norm=encoder_norm,
                                family=family, reducer_mode=reducer_mode)
        model.reducer.fit(Xtr, ytr, seed=seed)
        for p in model.parameters():
            p.requires_grad_(True)
    else:
        model = copy.deepcopy(model)
        # Freeze everything, then unfreeze exactly what this adaptation mode allows. The
        # source reducer's StandardScaler is NOT refit -- applying the source normalisation
        # to the target cohort is what "transfer" means here, and refitting it would quietly
        # turn the experiment into per-dataset recentering.
        for p in model.parameters():
            p.requires_grad_(False)
        if adapt_mode == "readout":
            tgt = ["out.", "mlp.2."]
        elif adapt_mode == "tail":
            tgt = ["out.", "mlp.", "q."]
        else:                                   # tail_plus_projection
            tgt = ["out.", "mlp.", "q.", "reducer."]
        for n, p in model.named_parameters():
            if any(n.startswith(t) for t in tgt):
                p.requires_grad_(True)
        if not any(p.requires_grad for p in model.parameters()):
            raise RuntimeError(f"adapt_mode {adapt_mode!r} unfroze nothing")

    Ztr = model.reducer.transform_np(Xtr)
    train_adapter(model, Ztr, ytr, epochs=epochs or EPOCHS, lr=lr or LR, batch=BATCH,
                  beta=ANGLE_L2_BETA, weight_decay=PROJ_WEIGHT_DECAY,
                  alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, seed=seed)
    return model, model.reducer


def score_arm(arm, fitted, helper, Xte, yte):
    if arm == "linear_probe":
        p = fitted.predict_proba(helper.transform(Xte))[:, 1]
    elif helper is None:                      # spatial arm: maps go in raw
        p = predict_proba(fitted, Xte)
    else:
        p = predict_proba(fitted, helper.transform_np(Xte))
    yte = np.asarray(yte).astype(int)
    out = {"auroc": fast_auroc(yte, p), "ap": fast_auprc(yte, p),
           "prevalence": float(yte.mean())}
    try:
        cal = calibration_metrics(yte, p)
        if isinstance(cal, dict):
            out.update({f"cal_{k}": v for k, v in cal.items()})
    except Exception:
        pass
    return out, p


def pretanh_magnitude(model, X):
    """Median |pre-tanh| on X. This is the x-axis of the saturation figure and the number
    that says whether an arm's angles are still moving with the input."""
    if model is None:
        return float("nan")
    if not hasattr(model, "reducer"):
        # Spatial arm: the pre-tanh value is the output of its own proj (conv + BatchNorm).
        model.eval()
        with torch.no_grad():
            z = model.proj(torch.as_tensor(X, dtype=torch.float32))
            return float(torch.median(torch.abs(z)).item())
    model.eval()
    with torch.no_grad():
        Z = torch.as_tensor(model.reducer.transform_np(X), dtype=torch.float32)
        z = model.reducer(Z)
        return float(torch.median(torch.abs(z)).item())


print("overnight primitives ready:",
      "fixed_eval_split, draw_budget, make_pooled_arm, fit_arm, score_arm, pretanh_magnitude")


In [ ]:
# ==============================================================================================
# SW-1. LEARNING-RATE AND WEIGHT-DECAY SWEEP  [resumable]
# ==============================================================================================
# THE OBJECTION THIS ANSWERS. Every saturation result in this project was obtained at one
# optimiser setting: AdamW at lr = 0.05 with weight decay 1e-3 on the projection. The obvious
# reviewer question is whether angle-encoder saturation is a real failure mode or simply a
# learning rate chosen too high. Without this sweep the honest answer is "we do not know",
# and that is the single largest hole in the paper.
#
# The mechanism makes a testable prediction rather than a vague one. Under AdamW the per-step
# update to a weight is bounded by roughly the learning rate, so after n steps the projection
# norm can grow by order lr * n. Saturation should therefore appear at a learning rate
# proportional to 1/steps, and the pre-tanh magnitude should scale roughly linearly with the
# learning rate at fixed epoch budget. If that is what the sweep shows, the mechanism is
# confirmed quantitatively and the claim becomes "saturation occurs whenever lr * epochs is
# large enough", which is a statement a reader can apply to their own setting. If pre-tanh
# stays small at every learning rate except 0.05, the finding is much narrower and the paper
# must say so.
#
# Either outcome is worth having before submission. This is the experiment that converts the
# paper's biggest vulnerability into either a strength or an honest boundary.
import itertools
import json
import os
import time

import numpy as np

SW_LOG = os.path.join(OUTPUT_DIR, "sweep_lr_wd.jsonl")
LR_GRID = [0.001, 0.005, 0.01, 0.05, 0.1]
WD_GRID = [1e-3, 1e-2]
SW_NORMS = [False, True]
SW_SEEDS = [12345, 1, 2]
SW_BACKBONES = ["B1_swinunetr_suprem"]     # add B2/B3 on a second pass
SW_FAMILY = "Q1_sel"
SW_ARMS = ["quantum"]                       # the classical arm saturates too; add it if time
SW_WALL_MIN = 300


def sw_done():
    d = {}
    if os.path.exists(SW_LOG):
        with open(SW_LOG) as fh:
            for line in fh:
                line = line.strip()
                if line:
                    try:
                        r = json.loads(line)
                    except Exception:
                        continue
                    if r.get("key"):
                        d[r["key"]] = r
    return d


SW_DONE = sw_done()
_t0 = time.time()
_cells = list(itertools.product(SW_BACKBONES, LR_GRID, WD_GRID, SW_NORMS, SW_SEEDS, SW_ARMS))
print("=" * 94)
print(f"LEARNING-RATE / WEIGHT-DECAY SWEEP  --  {len(_cells)} cell(s), "
      f"{len(SW_DONE)} already done")
print(f"  lr {LR_GRID}   wd {WD_GRID}   norm {SW_NORMS}   seeds {SW_SEEDS}")
print("=" * 94)

for _b, _lr, _wd, _nm, _sd, _arm in _cells:
    _key = f"SW|{_b}|lr{_lr}|wd{_wd}|norm{int(_nm)}|s{_sd}|{_arm}"
    if _key in SW_DONE or _b not in X_BY_BACKBONE:
        continue
    if (time.time() - _t0) / 60.0 > SW_WALL_MIN - 3:
        print("  budget spent -- stopping cleanly. Re-run to continue.")
        break
    _X, _y = X_BY_BACKBONE[_b], TARGETS[PRIMARY_TARGET]
    _pool, _ev = fixed_eval_split(_y, GROUPS_ALL, frac=HOLDOUT_FRAC, seed=SEED)
    _t = time.time()
    try:
        _m = make_pooled_arm(_arm, _X.shape[1], _sd, encoder_norm=_nm,
                             family=SW_FAMILY, reducer_mode="trainable")
        _m.reducer.fit(_X[_pool], _y[_pool], seed=_sd)
        for _p in _m.parameters():
            _p.requires_grad_(True)
        _Z = _m.reducer.transform_np(_X[_pool])
        # weight decay is the swept quantity, so train_adapter is called directly rather
        # than through fit_arm, which would apply the project-wide PROJ_WEIGHT_DECAY.
        train_adapter(_m, _Z, _y[_pool], epochs=EPOCHS, lr=_lr, batch=BATCH,
                      beta=ANGLE_L2_BETA, weight_decay=_wd, alpha=FOCAL_ALPHA,
                      gamma=FOCAL_GAMMA, seed=_sd)
        _pre = pretanh_magnitude(_m, _X[_pool])
        _sc, _ = score_arm(_arm, _m, _m.reducer, _X[_ev], _y[_ev])
        with open(SW_LOG, "a") as _fh:
            _fh.write(json.dumps({"key": _key, "backbone": _b, "lr": _lr, "wd": _wd,
                                  "encoder_norm": bool(_nm), "seed": _sd, "arm": _arm,
                                  "pretanh": _pre, "minutes": (time.time() - _t) / 60.0,
                                  **_sc}, default=float) + "\n")
        print(f"  lr={_lr:<6} wd={_wd:<6} norm={int(_nm)} s={_sd:<6} "
              f"AUROC {_sc['auroc']:.3f}  pre-tanh {_pre:8.2f}  "
              f"({(time.time() - _t) / 60.0:.1f} min)")
    except Exception as _e:
        with open(SW_LOG, "a") as _fh:
            _fh.write(json.dumps({"key": _key, "error": f"{type(_e).__name__}: {_e}"}) + "\n")
        print(f"  {_key} FAILED: {type(_e).__name__}: {_e}")


In [ ]:
# ==============================================================================================
# SW-2. READ THE SWEEP -- does the mechanism predict where saturation appears?
# ==============================================================================================
import pandas as pd

_rows = [r for r in sw_done().values() if r.get("auroc") is not None]
if not _rows:
    print("no sweep results yet.")
else:
    _d = pd.DataFrame(_rows)
    _d.to_csv(os.path.join(OUTPUT_DIR, "sweep_lr_wd.csv"), index=False)
    print(f"{len(_d):,} cell(s) -> {os.path.join(OUTPUT_DIR, 'sweep_lr_wd.csv')}\n")

    print("=" * 94)
    print("MEDIAN PRE-TANH MAGNITUDE  (rows = learning rate, cols = weight decay x norm)")
    print("=" * 94)
    print(_d.pivot_table(index="lr", columns=["wd", "encoder_norm"],
                         values="pretanh", aggfunc="median").round(2).to_string())
    print("\n" + "=" * 94)
    print("AUROC, mean over seeds")
    print("=" * 94)
    print(_d.pivot_table(index="lr", columns=["wd", "encoder_norm"],
                         values="auroc", aggfunc="mean").round(4).to_string())

    _u = _d[_d["encoder_norm"] == False]                                      # noqa: E712
    _c = _d[_d["encoder_norm"] == True]                                       # noqa: E712
    print("\n" + "=" * 94)
    print("WHAT TO CONCLUDE")
    print("=" * 94)
    if len(_u) and len(_c):
        _g = _u.groupby("lr")["pretanh"].median()
        print(f"  unconditioned pre-tanh by lr: "
              + ", ".join(f"lr={k}: {v:.1f}" for k, v in _g.items()))
        # the mechanism predicts pre-tanh roughly proportional to lr at fixed epochs
        if len(_g) >= 3:
            _r = np.corrcoef(np.log(_g.index.values.astype(float)),
                             np.log(np.maximum(_g.values, 1e-9)))[0, 1]
            print(f"  log(pre-tanh) vs log(lr): r = {_r:.3f}")
            print("  The mechanism predicts a positive, roughly linear relationship on log-log")
            print("  axes. A strong positive r supports 'saturation appears once lr x epochs is")
            print("  large enough', which is a statement a reader can apply to their own setting.")
        _safe = sorted(_g[_g < 5].index)
        _sat = sorted(_g[_g >= 5].index)
        if _safe and _sat:
            print(f"  stays out of saturation : lr {_safe}")
            print(f"  saturates               : lr {_sat}")
            print(f"  -> scope every claim in the paper to lr >= {min(_sat)} at this epoch")
            print("     budget, and report the threshold. A reader can then check their own")
            print("     setting against it, which is more useful than an unscoped claim.")
        elif _safe:
            print(f"  NOTHING saturated at any learning rate tested ({_safe}).")
            print("  -> the saturation result does not reproduce in this configuration. Find")
            print("     out what differs from the main run (epoch budget is the first")
            print("     suspect) before the paper claims a mechanism.")
        else:
            print(f"  EVERY learning rate tested saturated ({_sat}), down to the smallest.")
            print("  -> the effect is not an artefact of one setting, and the paper can say so")
            print("     with a sweep behind it. This is the strongest available outcome.")
        _best_u = _u.groupby("lr")["auroc"].mean().max()
        _best_c = _c.groupby("lr")["auroc"].mean().max()
        print(f"\n  best unconditioned AUROC over the whole grid : {_best_u:.4f}")
        print(f"  best conditioned   AUROC over the whole grid : {_best_c:.4f}")
        print(f"  difference: {_best_c - _best_u:+.4f}")
        print("  THIS IS THE NUMBER THAT MATTERS. If a well-tuned unconditioned model reaches")
        print("  the conditioned model's performance, the contribution is a tuning shortcut. If")
        print("  it does not, conditioning buys something tuning cannot, and that belongs in the")
        print("  abstract.")


In [ ]:
# ==============================================================================================
# SW-3. QUBIT WIDTH, AND A DIRECT TEST OF THE SIGN-CODE CLAIM  [resumable]
# ==============================================================================================
# TWO OBJECTIONS THIS ANSWERS, AND THEY ARE BOTH FAIR ONES.
#
# (1) The paper says a saturated encoder "collapses to a discrete sign code of at most 2^n
#     patterns". As written that is an inference from the pre-tanh magnitude, not a
#     measurement. It is directly measurable. Count the sign patterns the encoder actually
#     realises on the evaluation set; measure how much the angles still vary WITHIN one
#     pattern; and ask whether a logistic regression that sees ONLY the sign bits reproduces
#     the quantum arm's own AUROC. If it does, the circuit is demonstrably operating as a
#     lookup table over the code, and the mechanism stops being a story about tanh and
#     becomes an observation.
#
# (2) Every number in the paper is at n = 4 qubits. If the damage is really the loss of
#     continuous magnitude down to a 2^n code, then it must depend on n: a 2-qubit encoder
#     that saturates has 4 cells to represent the whole feature vector, an 8-qubit one has
#     256. The prediction is that the conditioning gain SHRINKS as n grows. A reviewer who
#     asks "is this just an artefact of your qubit count?" is asking the right question, and
#     a width sweep is the one-figure answer. If instead the gain is flat in n, the paper
#     must say that too -- it would mean the loss is not principally about code size.
#
# The cost is three widths x two conditionings x three seeds on one backbone. Q4's tree
# ansatz needs a power of two, so 2/4/8 is also the natural grid.
import json
import os
import time
from contextlib import contextmanager

import numpy as np
import torch

SW3_LOG = os.path.join(OUTPUT_DIR, "sweep_qubits.jsonl")
NQ_GRID = [2, 4, 8]
SW3_NORMS = [False, True]
SW3_SEEDS = [12345, 1, 2]
SW3_BACKBONES = ["B1_swinunetr_suprem"]
SW3_FAMILY = "Q1_sel"
SW3_ARM = "quantum"
SW3_WALL_MIN = 240


@contextmanager
def qubit_width(n):
    """Temporarily run the whole adapter factory at a different width.

    make_pooled_arm reads OV_CFG at call time, so this is the least invasive way to change
    the width without a second copy of the constructor drifting out of sync with the first.
    Restored in a finally block: a raised exception inside the loop must not leave the rest
    of the notebook running at the wrong width.
    """
    old = OV_CFG["n_qubits"]
    OV_CFG["n_qubits"] = int(n)
    try:
        yield int(n)
    finally:
        OV_CFG["n_qubits"] = old


def sign_code_report(model, X_fit, y_fit, X_ev, y_ev):
    """Measure directly whether the encoder has collapsed to a discrete sign code.

    Returns the three quantities that together settle it: how many of the 2^n patterns are
    realised, how much of the angle variation survives inside a pattern, and how well a
    classifier restricted to the sign bits alone does. A saturated encoder gives a small
    effective code count, a within-code ratio near zero, and a sign-bit probe that matches
    the arm it was taken from. A responsive encoder gives the opposite on all three.
    """
    out = {}
    if model is None or not hasattr(model, "reducer") or model.reducer.proj is None:
        return out
    model.eval()
    with torch.no_grad():
        _f = torch.as_tensor(model.reducer.transform_np(X_fit), dtype=torch.float32)
        _e = torch.as_tensor(model.reducer.transform_np(X_ev), dtype=torch.float32)
        zf = model.reducer(_f).numpy()
        ze = model.reducer(_e).numpy()
    n = int(ze.shape[1])
    a_ev = (np.pi / 2.0) * np.tanh(ze)
    out["n_qubits"] = n
    out["pretanh"] = float(np.median(np.abs(ze)))
    # |tanh| > 1 - 1e-6 is |z| > 7.25: the region where the angle no longer resolves the input
    out["sat_frac"] = float(np.mean(np.abs(np.tanh(ze)) > 1.0 - 1e-6))

    pw = (1 << np.arange(n)).astype(np.int64)
    bits_f = (zf > 0).astype(np.int64)
    bits_e = (ze > 0).astype(np.int64)
    code_f, code_e = bits_f @ pw, bits_e @ pw
    _u, _c = np.unique(code_e, return_counts=True)
    _p = _c / _c.sum()
    out["codes_seen"] = int(len(_u))
    out["codes_possible"] = int(2 ** n)
    # effective number of codes = exp(entropy): 100 series split 99/1 across two patterns is
    # one code in practice, and the raw count would call it two.
    out["codes_effective"] = float(np.exp(-(_p * np.log(_p)).sum()))

    # How much angle variation survives INSIDE a single sign pattern? A true sign code has
    # none: every series in a cell receives identical angles.
    _tot = float(np.mean(np.std(a_ev, axis=0)))
    _within = [float(np.mean(np.std(a_ev[code_e == c], axis=0)))
               for c in _u if int((code_e == c).sum()) >= 5]
    out["angle_within_code_ratio"] = (float(np.mean(_within) / _tot)
                                      if _within and _tot > 0 else float("nan"))

    y_fit = np.asarray(y_fit).astype(int)
    y_ev = np.asarray(y_ev).astype(int)
    # (a) the sign bits as ordinary features
    try:
        _lr = LogisticRegression(max_iter=2000).fit(bits_f * 2 - 1, y_fit)
        out["auroc_sign_bits"] = fast_auroc(y_ev, _lr.predict_proba(bits_e * 2 - 1)[:, 1])
    except Exception:
        out["auroc_sign_bits"] = float("nan")
    # (b) the full lookup table over patterns -- the strongest form of the claim, and the
    #     upper bound on anything a circuit can compute from a saturated encoder
    try:
        _seen = {int(c): i for i, c in enumerate(np.unique(code_f))}
        if len(_seen) <= 64:
            H_f = np.zeros((len(code_f), len(_seen)), dtype=np.float32)
            H_f[np.arange(len(code_f)), [_seen[int(c)] for c in code_f]] = 1.0
            H_e = np.zeros((len(code_e), len(_seen)), dtype=np.float32)
            for i, c in enumerate(code_e):
                j = _seen.get(int(c))
                if j is not None:
                    H_e[i, j] = 1.0
            _lut = LogisticRegression(max_iter=2000).fit(H_f, y_fit)
            out["auroc_code_lut"] = fast_auroc(y_ev, _lut.predict_proba(H_e)[:, 1])
        else:
            out["auroc_code_lut"] = float("nan")
    except Exception:
        out["auroc_code_lut"] = float("nan")
    return out


def sw3_done():
    d = {}
    if os.path.exists(SW3_LOG):
        with open(SW3_LOG) as fh:
            for line in fh:
                line = line.strip()
                if line:
                    try:
                        r = json.loads(line)
                    except Exception:
                        continue
                    if r.get("key"):
                        d[r["key"]] = r
    return d


SW3_DONE = sw3_done()
_t0 = time.time()
_cells3 = [(b, n, m, s) for b in SW3_BACKBONES for n in NQ_GRID
           for m in SW3_NORMS for s in SW3_SEEDS]
print("=" * 94)
print(f"QUBIT-WIDTH SWEEP + SIGN-CODE DIAGNOSTIC  --  {len(_cells3)} cell(s), "
      f"{len(SW3_DONE)} already done")
print(f"  n {NQ_GRID}   norm {SW3_NORMS}   seeds {SW3_SEEDS}   lr {LR}   family {SW3_FAMILY}")
print("=" * 94)

for _b, _nq, _nm, _sd in _cells3:
    _key = f"SW3|{_b}|n{_nq}|norm{int(_nm)}|s{_sd}"
    if _key in SW3_DONE or _b not in X_BY_BACKBONE:
        continue
    if (time.time() - _t0) / 60.0 > SW3_WALL_MIN - 3:
        print("  budget spent -- stopping cleanly. Re-run to continue.")
        break
    _X, _y = X_BY_BACKBONE[_b], TARGETS[PRIMARY_TARGET]
    _pool, _ev = fixed_eval_split(_y, GROUPS_ALL, frac=HOLDOUT_FRAC, seed=SEED)
    _t = time.time()
    try:
        with qubit_width(_nq):
            _m = make_pooled_arm(SW3_ARM, _X.shape[1], _sd, encoder_norm=_nm,
                                 family=SW3_FAMILY, reducer_mode="trainable")
            _m.reducer.fit(_X[_pool], _y[_pool], seed=_sd)
            for _p in _m.parameters():
                _p.requires_grad_(True)
            _Z = _m.reducer.transform_np(_X[_pool])
            train_adapter(_m, _Z, _y[_pool], epochs=EPOCHS, lr=LR, batch=BATCH,
                          beta=ANGLE_L2_BETA, weight_decay=PROJ_WEIGHT_DECAY,
                          alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, seed=_sd)
            _sc, _ = score_arm(SW3_ARM, _m, _m.reducer, _X[_ev], _y[_ev])
            _diag = sign_code_report(_m, _X[_pool], _y[_pool], _X[_ev], _y[_ev])
        _rec = {"key": _key, "backbone": _b, "nq": int(_nq), "encoder_norm": bool(_nm),
                "seed": _sd, "arm": SW3_ARM, "family": SW3_FAMILY,
                "minutes": (time.time() - _t) / 60.0, **_sc, **_diag}
        with open(SW3_LOG, "a") as _fh:
            _fh.write(json.dumps(_rec, default=float) + "\n")
        print(f"  n={_nq} norm={int(_nm)} s={_sd:<6} AUROC {_sc['auroc']:.3f}  "
              f"pre-tanh {_diag.get('pretanh', float('nan')):7.2f}  "
              f"codes {_diag.get('codes_seen', 0)}/{_diag.get('codes_possible', 0)} "
              f"(eff {_diag.get('codes_effective', float('nan')):.1f})  "
              f"sign-bit AUROC {_diag.get('auroc_sign_bits', float('nan')):.3f}  "
              f"({(time.time() - _t) / 60.0:.1f} min)")
    except Exception as _e:
        with open(SW3_LOG, "a") as _fh:
            _fh.write(json.dumps({"key": _key, "error": f"{type(_e).__name__}: {_e}"}) + "\n")
        print(f"  {_key} FAILED: {type(_e).__name__}: {_e}")


In [ ]:
# ==============================================================================================
# SW-4. READ THE WIDTH SWEEP -- is the collapse really a sign code, and does it depend on n?
# ==============================================================================================
import numpy as np
import pandas as pd

_rows3 = [r for r in sw3_done().values() if r.get("auroc") is not None]
if not _rows3:
    print("no width-sweep results yet -- run SW-3 first.")
else:
    _d3 = pd.DataFrame(_rows3)
    _d3["encoder_norm"] = _d3["encoder_norm"].astype(bool)
    _csv3 = os.path.join(OUTPUT_DIR, "sweep_qubits.csv")
    _d3.to_csv(_csv3, index=False)
    print(f"{len(_d3):,} cell(s) -> {_csv3}\n")

    _cols = [c for c in ["auroc", "pretanh", "sat_frac", "codes_seen", "codes_effective",
                         "angle_within_code_ratio", "auroc_sign_bits", "auroc_code_lut"]
             if c in _d3.columns]
    print("=" * 94)
    print("BY WIDTH AND CONDITIONING  (mean over seeds)")
    print("=" * 94)
    _tab = _d3.groupby(["nq", "encoder_norm"])[_cols].mean().round(4)
    print(_tab.to_string())

    _u3 = _d3[_d3["encoder_norm"] == False]                                   # noqa: E712
    _c3 = _d3[_d3["encoder_norm"] == True]                                    # noqa: E712

    print("\n" + "=" * 94)
    print("IS THE SATURATED ENCODER A SIGN CODE?   (unconditioned cells only)")
    print("=" * 94)
    if len(_u3):
        for _nq, _g in _u3.groupby("nq"):
            _eff = _g["codes_effective"].mean() if "codes_effective" in _g else float("nan")
            _rat = _g["angle_within_code_ratio"].mean() if "angle_within_code_ratio" in _g else float("nan")
            _a, _sb = _g["auroc"].mean(), _g.get("auroc_sign_bits", pd.Series([np.nan])).mean()
            _lut = _g.get("auroc_code_lut", pd.Series([np.nan])).mean()
            print(f"  n = {int(_nq)}: effective codes {_eff:5.2f} of {2 ** int(_nq):<4} | "
                  f"within-code angle spread {_rat:6.3f} of total | "
                  f"arm {_a:.3f}  sign-bits {_sb:.3f}  lookup-table {_lut:.3f}")
        print()
        print("  READ IT LIKE THIS. A within-code angle spread near zero means every series")
        print("  sharing a sign pattern receives the SAME angles -- the encoder has stopped")
        print("  transmitting magnitude and is transmitting only the pattern. If in addition")
        print("  the sign-bit or lookup-table AUROC matches the arm's own, then everything the")
        print("  circuit achieves is available from the code alone, and the claim in the paper")
        print("  is a measurement rather than an interpretation. Report the largest gap between")
        print("  the arm and its lookup table as the honest residual.")
        if {"auroc", "auroc_code_lut"} <= set(_u3.columns):
            _gap = (_u3["auroc"] - _u3["auroc_code_lut"]).abs()
            if _gap.notna().any():
                print(f"\n  largest |arm - lookup table| over unconditioned cells: {_gap.max():.4f}")
                print(f"  mean:                                                  {_gap.mean():.4f}")

    print("\n" + "=" * 94)
    print("DOES THE DAMAGE DEPEND ON THE WIDTH?")
    print("=" * 94)
    if len(_u3) and len(_c3):
        _gu = _u3.groupby("nq")["auroc"].mean()
        _gc = _c3.groupby("nq")["auroc"].mean()
        _gain = (_gc - _gu).dropna()
        for _nq, _v in _gain.items():
            print(f"  n = {int(_nq)}: unconditioned {_gu[_nq]:.4f}  conditioned {_gc[_nq]:.4f}"
                  f"  gain {_v:+.4f}")
        if len(_gain) >= 3:
            _r = np.corrcoef(np.log2(_gain.index.values.astype(float)), _gain.values)[0, 1]
            print(f"\n  correlation of the conditioning gain with log2(n): r = {_r:.3f}")
            if _r < -0.5:
                print("  -> the gain SHRINKS as the code widens, which is what the mechanism")
                print("     predicts: a wider code destroys less. The paper can say the effect")
                print("     is a property of the encoder's information bottleneck, and scope")
                print("     it to narrow encoders, which is where hybrid work actually sits.")
            elif _r > 0.5:
                print("  -> the gain GROWS with n, which the sign-code account does not predict.")
                print("     Do not report the width sweep as supporting the mechanism; report it")
                print("     as an open observation and say so plainly.")
            else:
                print("  -> the gain is roughly flat in n over the range tested. The failure is")
                print("     therefore not principally about how many cells the code has; it is")
                print("     about the loss of magnitude itself. Say that, and drop any wording")
                print("     that implies the 2^n count is doing the work.")
        print("\n  Widths tested are 2, 4 and 8. Any claim about n beyond that range is")
        print("  extrapolation and should not appear in the paper.")
